# Problems writeup

## Byte-Pair Encoding (BPE) Tokenizer

### The Unicode Standard

**Problem (**`unicode1`**): Understanding Unicode (1 point)**



- What Unicode character does `chr(0)` return?

  > It returns the character named `NUL`

- How does this character's string representation (`__repr__()`) differ from its printed representation?
  > The `__repr__` is the encoding of the printed representation 
  > with printable characters, especially with the backslash escaped
  > as well as the single quotes included.

In [2]:
print(
    f"chr(0)->[{chr(0)}]\n"
    f"chr(0).__repr__()->[{chr(0).__repr__()}]"
)
chr(0).__repr__()

chr(0)->[ ]
chr(0).__repr__()->['\x00']


"'\\x00'"

- What happens when this character occurs in text? 
  It may be helpful to play around with 
  the following in your Python interpreter 
  and see if it matches your expectations:

In [3]:
chr(0)

'\x00'

In [4]:
print(chr(0))

 


In [5]:
"this is a test" + chr(0) + "string"

'this is a test\x00string'

In [6]:
if not print("this is a test" + chr(0) + "string", sep="+"):
    from io import StringIO
    with StringIO() as s_:
        print("this is a test" + chr(0) + "string", file=s_)
        s = s_.getvalue()
s

this is a test string


'this is a test\x00string\n'

In [7]:
print(s)

this is a test string



The tests reveal that a string containing `NUL` 
get treated variously by different front ends; 
Jupyter displays it as a whitespace
while Python REPL simply omits it.
Neither treats it the C way, though Windows clipboard API does.

### Unicode Encodings

**Problem (**`unicode2`**): Unicode Encodings (3 points)**

- What are some reasons to prefer 
  training our tokenizer on UTF-8 encoded bytes, 
  rather than UTF-16 or UTF-32? 
  It may be helpful to compare the output of these encodings for various input strings.

  > UTF32 is constant length encoding but too wasteful for ASCII; UTF16 is legacy cringe.

- Consider the following (incorrect) function, 
  which is intended to decode a UTF-8 byte string into a Unicode string.
  Why is this function incorrect? 
  Provide an example of an input byte string that yields incorrect results.

In [8]:
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])
decode_utf8_bytes_to_str_wrong("hello".encode("utf-8"))

'hello'

Not every UT8 code unit (i.e. a 8-bit byte) is a encodes a Unicode code point.

See [[Accepted] What's the difference between a character, a code point, a glyph and a grapheme? - Stack Overflow](https://stackoverflow.com/questions/27331819/whats-the-difference-between-a-character-a-code-point-a-glyph-and-a-grapheme/27331885#27331885)

In [9]:
import sys, traceback  # noqa: E401
try:
    decode_utf8_bytes_to_str_wrong("こんにちは".encode())
except UnicodeDecodeError:
    traceback.print_exc(file=sys.stdout)

Traceback (most recent call last):
  File "/tmp/ipykernel_59400/1442419784.py", line 3, in <module>
    decode_utf8_bytes_to_str_wrong("こんにちは".encode())
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_59400/488356226.py", line 2, in decode_utf8_bytes_to_str_wrong
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])
                    ~~~~~~~~~~~~~~~~~^^^^^^^^^
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe3 in position 0: unexpected end of data


- Give a two-byte sequence that does not decode to any Unicode character(s).

In [10]:
for b in [
        b"\x80\x80",     # lone continuation
        b"\xE2\x82",     # missing continuation
        b"\xC1\xBF",     # forbidden lead - overlong encoding of U+007F
        b"\xF0\x80",     # overlong encoding of U+0000
        b"\xF5\x80",     # out of range
    ]:
    try:
        print(b.decode("utf-8"))
    except UnicodeDecodeError as e:
        print("\n", e.object, e.__traceback__, flush=True)
        traceback.print_exc(file=sys.stdout)


 b'\x80\x80' <traceback object at 0x7854583ee2c0>
Traceback (most recent call last):
  File "/tmp/ipykernel_59400/3489787588.py", line 9, in <module>
    print(b.decode("utf-8"))
          ~~~~~~~~^^^^^^^^^
UnicodeDecodeError: 'utf-8' codec can't decode byte 0x80 in position 0: invalid start byte

 b'\xe2\x82' <traceback object at 0x7854583ee2c0>
Traceback (most recent call last):
  File "/tmp/ipykernel_59400/3489787588.py", line 9, in <module>
    print(b.decode("utf-8"))
          ~~~~~~~~^^^^^^^^^
UnicodeDecodeError: 'utf-8' codec can't decode bytes in position 0-1: unexpected end of data

 b'\xc1\xbf' <traceback object at 0x7854583ee2c0>
Traceback (most recent call last):
  File "/tmp/ipykernel_59400/3489787588.py", line 9, in <module>
    print(b.decode("utf-8"))
          ~~~~~~~~^^^^^^^^^
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc1 in position 0: invalid start byte

 b'\xf0\x80' <traceback object at 0x78545828c580>
Traceback (most recent call last):
  File "/tmp/ip

### Sub-word Tokenization

Sub-word tokenization is a midpoint between word-level tokenizers and byte-level tokenizers.

In this assignment, we'll implement a byte-level BPE tokenizer.  
The process of constructing the BPE tokenizer vocabulary 
is known as "training" the BPE tokenizer.

### BPE Tokenizer Training

- Vocabulary initialization
- Pre-tokenization
  - The original BPE implementation pre-tokenizes by simply splitting on whitespace
  - Most modern tokenizers use a regex-based pre-tokenizer, like

In [6]:
# import base64
import requests

# retrieved from <https://github.com/openai/tiktoken/pull/234/changes>
url: str = "https://github.com/l0rinc/tiktoken/blob/6f261deef63b49a7da9000b57a7cf938d7315ab3/tiktoken_ext/openai_public.py#L23"
parts= url.split("github.com/", 1)[1].split("/")
owner: str = parts[0]
repo: str = parts[1]
ref_id: str = parts[3]
path: str = "/".join(parts[4:]).split("#", 1)[0]
# extract line number or line number range
# ln_no: int = int(parts[-1].split("#L", 1)[1])
lns= parts[-1].split("#L", 1)[1].split("-", 1)
ln_start: int = int(lns[0])
ln_end: int = int(lns[1]) if len(lns) > 1 else int(lns[0])

api: str = f"https://api.github.com/repos/{owner}/{repo}/contents/{path}"
raw: str = f"https://raw.githubusercontent.com/{owner}/{repo}/{ref_id}/{path}"
params: dict[str, str] = {"ref": ref_id}

# data = requests.get(api, params=params).json() # rate limited, will fail for public proxy
# content_api = base64.b64decode(data["content"]).decode("utf-8")
# line_23_api = content_api.splitlines()[22]

text = requests.get(raw).text
lines = text.splitlines()[ln_start - 1:ln_end]
[print(line) for line in lines]
# text.splitlines()[ln_start - 1:ln_end]

# split json key and value by the first colon, and strip whitespace
kv: dict[str, str] = dict(line.split(":", 1) for line in lines)
kv = dict((k.strip(), v.strip()) for k, v in kv.items())

# oh no, we need to create a regex literal from a string literal
# giving up on this nightmare of security folks
PAT: str = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

        "pat_str": r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""",


In [10]:
# requires `regex` package
import regex as re
re.findall(PAT, "some text that i'll pre-tokenize")
[match.group() for match in
    re.finditer(PAT, "some text that i'll pre-tokenize")]

['some', ' text', ' that', ' i', "'ll", ' pre', '-', 'tokenize']

- Compute BPE merges

### Experimenting with BPE Tokenizer Training

In [31]:
%load_ext autoreload
%autoreload 2

In [ ]:
%%bash # fetch assignment datasets
set -euo pipefail

mkdir -p data/ && cd data/
mkdir -p bak/  && mv -b * bak/ 2>/dev/null || true

touch .gitignore && echo "*" > .gitignore

wget --show-progress --progress=bar:force:noscroll \
    https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt
wget --show-progress --progress=bar:force:noscroll \
    https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-valid.txt

wget --show-progress --progress=bar:force:noscroll \
    https://huggingface.co/datasets/stanford-cs336/owt-sample/resolve/main/owt_train.txt.gz
gunzip owt_train.txt.gz
wget --show-progress --progress=bar:force:noscroll \
    https://huggingface.co/datasets/stanford-cs336/owt-sample/resolve/main/owt_valid.txt.gz
gunzip owt_valid.txt.gz

cd ..

#### 
**Problem (**`train_bpe`**): BPE Tokenizer Training (15 points)**

In [15]:
%uv --directory .. run pytest -vv tests/test_train_bpe.py

============================= test session starts ==============================
platform linux -- Python 3.13.13, pytest-9.0.2, pluggy-1.6.0 -- /workspaces/stanford-cs336.assignment1-basics/.venv/bin/python3
cachedir: .pytest_cache
rootdir: /workspaces/stanford-cs336.assignment1-basics
configfile: pyproject.toml
plugins: jaxtyping-0.3.9, timeout-2.4.0
collected 3 items                                                              

tests/test_train_bpe.py::test_train_bpe_speed PASSED
tests/test_train_bpe.py::test_train_bpe PASSED
tests/test_train_bpe.py::test_train_bpe_special_tokens Training BPE with 20 processes...
[███│███│███│███│███│██│███│███│███│███│██│███│███│███│███│██│███│███│███│███]
PASSED

=============================== warnings summary ===============================
tests/test_train_bpe.py: 20 warnings
  /usr/local/lib/python3.13/multiprocessing/popen_fork.py:67: DeprecationWarning: This process (pid=404192) is multi-threaded, use of fork() may lead to deadlocks in the 

In [ ]:
# scratchpad

# prototyping the multiprocessing parameters
for cpu in [None, 0, 1, 2, 4, 20]:
    ns = [cpu or x or 1 for x in [-1, None, False, 0, True, 1, 2, 1000]]
    print([max(1, min(n if x is True else x or 1, n or 1)) for x, n in zip([-1, None, False, 0, True, 1, 2, 1000], ns)])

# experimenting with str / bytes interactions
x = "hello world"
tuple(ch.encode() for ch in x)
b"Hello world"[2:4]
list(zip(x, x[1:]))
list(x.encode())

# experimenting with zip()
x, y = zip(*{1: 'a', 2: 'b'}.items())
print(x, y)

[1, 1, 1, 1, 1, 1, 2, 1000]
[1, 1, 1, 1, 1, 1, 2, 1000]
[1, 1, 1, 1, 1, 1, 1, 1]
[1, 1, 1, 1, 2, 1, 2, 2]
[1, 1, 1, 1, 4, 1, 2, 4]
[1, 1, 1, 1, 20, 1, 2, 20]


In [55]:
# test train_bpe on a simple string
import os
from importlib import reload
import cs336_basics.tokenizer
reload(cs336_basics.tokenizer)
train_bpe = cs336_basics.tokenizer.train_bpe
# create a small corpus file
with open("temp.txt", "w") as f:
    f.write("""low low low low low
lower lower widest widest widest
newest newest newest newest newest newest<|endoftext|>""")

f = "temp.txt"
f = "../tests/fixtures/tinystories_sample.txt"
vocab, merges = train_bpe(f, 256+20, ["<|endoftext|>"])
print([(id, vocab[id]) for id in range(256, len(vocab))])
print(merges)

# delete the temporary file
os.remove("temp.txt")

[(256, b'\xc3\xbf'), (257, b' t'), (258, b'he'), (259, b' a'), (260, b' s'), (261, b' w'), (262, b'nd'), (263, b'ed'), (264, b' f'), (265, b' the'), (266, b'it'), (267, b' to'), (268, b' d'), (269, b'is'), (270, b' b'), (271, b'll'), (272, b' wa'), (273, b' and'), (274, b'ou'), (275, b'ay')]
[(b' ', b't'), (b'h', b'e'), (b' ', b'a'), (b' ', b's'), (b' ', b'w'), (b'n', b'd'), (b'e', b'd'), (b' ', b'f'), (b' t', b'he'), (b'i', b't'), (b' t', b'o'), (b' ', b'd'), (b'i', b's'), (b' ', b'b'), (b'l', b'l'), (b' w', b'a'), (b' a', b'nd'), (b'o', b'u'), (b'a', b'y')]


In [ ]:
# test train_bpe on a 5M sample from TinyStories
from cs336_basics.tokenizer import train_bpe
f = "../tests/fixtures/tinystories_sample_5M.txt"
vocab, merges = train_bpe(f, 256+20, ["<|endoftext|>"])
print([(id, vocab[id]) for id in range(256, len(vocab))])
print(merges)

Training BPE with 20 processes...


[███│███│███│███│███│██│███│███│███│███│██│███│███│███│███│██│███│███│███│███]
[(256, b'\xff'), (257, b' s'), (258, b'in'), (259, b'ed'), (260, b'er'), (261, b'ing'), (262, b' c'), (263, b' b'), (264, b'es'), (265, b' p'), (266, b' t'), (267, b'ar'), (268, b'an'), (269, b' f'), (270, b're'), (271, b' w'), (272, b'en'), (273, b' h'), (274, b' d'), (275, b'le')]
[(b' ', b's'), (b'i', b'n'), (b'e', b'd'), (b'e', b'r'), (b'in', b'g'), (b' ', b'c'), (b' ', b'b'), (b'e', b's'), (b' ', b'p'), (b' ', b't'), (b'a', b'r'), (b'a', b'n'), (b' ', b'f'), (b'r', b'e'), (b' ', b'w'), (b'e', b'n'), (b' ', b'h'), (b' ', b'd'), (b'l', b'e')]


####
**Problem (**`train_bpe_tinystories`**):  BPE Training on TinyStories (2 points)**

(a) Train a byte-level BPE tokenizer on the TinyStories dataset, 
using a maximum vocabulary size of 10,000. 
Make sure to add the TinyStories `<|endoftext|>` special token to the vocabulary. 
Serialize the resulting vocabulary and merges to disk for further inspection. 
How much time and memory did training take? 
What is the longest token in the vocabulary? 
Does it make sense?

  **Resource requirements**: ≤ 30 minutes (no GPUs), ≤ 30 GB RAM

  **Hint**  You should be able to get under 2 minutes for BPE training using multiprocessing
  during pre-tokenization and the following two facts:  
    (a) The `<|endoftext|>` token delimits documents in the data files.  
    (b) The `<|endoftext|>` token is handled as a special case before the BPE merges are applied.

**Deliverable**: A one-to-two sentence response.

In [25]:
%time??

Source:
    @no_var_expand
    @magic_arguments.magic_arguments()
    @magic_arguments.argument(
        "--no-raise-error",
        action="store_true",
        dest="no_raise_error",
        help="If given, don't re-raise exceptions",
    )
    @magic_arguments.kwds(
        epilog="""
      Any remaining arguments will be treated as code to run.
    """
    )
    @skip_doctest
    @needs_local_scope
    @line_cell_magic
    @output_can_be_silenced
    def time(self, line="", cell=None, local_ns=None):
        """Time execution of a Python statement or expression.

        The CPU and wall clock times are printed, and the value of the
        expression (if any) is returned.  Note that under Win32, system time
        is always reported as 0, since it can not be measured.

        This function can be used both as a line and cell magic:

        - In line mode you can time a single-line statement (though multiple
          ones can be chained with using semicolons).

        - In cel

In [35]:
import time
import resource
from importlib import reload
from cs336_basics.tokenizer import train_bpe
from cs336_basics.common import prettyprint_vocab, PeakMemoryMonitor

f = "data/TinyStoriesV2-GPT4-train.txt"
ru0 = resource.getrusage(resource.RUSAGE_SELF)
t0 = time.perf_counter()
with PeakMemoryMonitor() as mem:
    vocab, _ = train_bpe(f, 10000, ["<|endoftext|>"])
wall_time = time.perf_counter() - t0
ru1 = resource.getrusage(resource.RUSAGE_SELF)
cpu_user = ru1.ru_utime - ru0.ru_utime
cpu_sys  = ru1.ru_stime - ru0.ru_stime
print(f"CPU times: user {cpu_user:.2f}s, sys: {cpu_sys:.2f}s, total: {cpu_user + cpu_sys:.2f}s")
print(f"Wall time: {wall_time:.1f}s")
print(f"Peak RAM (main + workers): {mem.peak_mb:.0f} MB")
prettyprint_vocab(vocab, cols=10, col_width=13)


Training BPE with 20 processes...
[███│███│███│███│███│██│███│███│███│███│██│███│███│███│███│██│███│███│███│███]
CPU times: user 4.90s, sys: 0.99s, total: 5.89s
Wall time: 66.5s
Peak RAM (main + workers): 8088 MB
    id │             0 │             1 │             2 │             3 │             4 │             5 │             6 │             7 │             8 │             9 │ 
─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   250 │             ù │             ú │             û │             ü │             ý │             þ │             ÿ │            in │            er │            ed │ 
   260 │            Ġs │           ing │            es │            ar │            Ġc │            an │            en │            at │            on │            Ġp │ 
   270 │            re │            Ġb │            or │            ll │            Ġt │            le │   

In [36]:
longest = ""
for token in vocab.values():
    try:
        new=token.decode()
    except UnicodeDecodeError:
        new = "?"
    if len(new) > len(longest):
        longest = new
print(longest)

!grep -m 10 -C 2 -nH "{longest}" "data/TinyStoriesV2-GPT4-train.txt"

ssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssss


data/TinyStoriesV2-GPT4-train.txt-7093599-"No, mine is faster!" Lily says. She pushes her car harder and it goes farther.
data/TinyStoriesV2-GPT4-train.txt-7093600-They keep pushing their cars until they reach the wall. Then they turn them around and push them again.
data/TinyStoriesV2-GPT4-train.txt:7093601:But then, Lily's car stops. It makes a funny sound. "Pssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssss

####
**Problem (**`train_bpe_expts_owt`**):  BPE Training on OpenWebText (2 points)**

In [ ]:
from cs336_basics.tokenizer import train_bpe
from cs336_basics.common import prettyprint_vocab

f = "data/TinyStoriesV2-GPT4-train.txt"
vocab, _ = train_bpe(f, 10000, ["<|endoftext|>"])
prettyprint_vocab(vocab, cols=10, col_width=13)

Training BPE with 20 processes...
[███│███│███│███│███│██│███│███│███│███│██│███│███│███│███│██│███│███│███│███]
    id │             0 │             1 │             2 │             3 │             4 │             5 │             6 │             7 │             8 │             9 │ 
─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   250 │             ù │             ú │             û │             ü │             ý │             þ │             ÿ │            in │            er │            ed │ 
   260 │            Ġs │           ing │            es │            ar │            Ġc │            an │            en │            at │            on │            Ġp │ 
   270 │            re │            Ġb │            or │            ll │            Ġt │            le │            is │            Ġd │            ou │            Ġf │ 
   280 │            as │            o